# Statistics — Multi-Seed Aggregation

For each HL-line and wave condition, globs **all available seed files** (e.g. `run1-1.h5`, `run1-2.h5`, …) and computes:

| Column | Definition |
|---|---|
| `static_kN` | Sum of FTotal across all 8 moorings (seed-independent) |
| `dyn_mean_kN` | Mean of per-seed means of the mooring summed time-series |
| `dyn_max_kN` | Absolute peak across all seeds — each metric uses its own governing seed |
| `dyn_std_kN` | Std of per-seed peaks (spread across seeds) |
| `n_runs` | Number of seed files found |

Each metric's `dyn_max_kN` is taken independently: system mooring peak from the seed with the highest mooring peak, system fender peak from the seed with the highest fender peak, each individual line/fender/DOF from its own governing seed.

Output: **`statistics.csv`** — consumed by `02_DAF_Moorings_System.ipynb`.

In [ ]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

HEADINGS     = ['0deg', '45deg', '90deg', '135deg', '180deg']
# Resolve this notebook's folder (works in VS Code and standard Jupyter)
NOTEBOOK_DIR = Path(globals().get('__vsc_ipynb_file__', Path.cwd() / '_')).parent
os.chdir(NOTEBOOK_DIR)
import sys
sys.path.insert(0, str(NOTEBOOK_DIR))
from thesis_style import apply, full
apply()
print(f'Working directory: {Path.cwd()}')

In [2]:
g         = 9.81
N_COND    = 10
HS_VALUES = np.linspace(0.5, 3.5, N_COND)

_S = 'BerthedTanker_Catenary'
_D = 'BerthedTanker_Catenary'

LINES = {
    'HL7':  {'n': 7,  'base_s': f'{_S}/Run1',  'base_d': f'{_D}/Run1',
             'static': 'HL7/StaticResults_HL7_run1-1.h5',
             'dynamic': 'HL7/DynamicResults_HL7_run1-1.h5',
             'n_cond': 9},
    'HL9':  {'n': 9,  'base_s': f'{_S}/Run2',  'base_d': f'{_D}/Run2',
             'static': 'HL9/StaticResults_HL9_run2-1.h5',
             'dynamic': 'HL9/DynamicResults_HL9_run2-1.h5',
             'n_cond': 9},
    'HL11': {'n': 11, 'base_s': f'{_S}/Run3',  'base_d': f'{_D}/Run3',
             'static': 'HL11/StaticResults_HL11_run3-1.h5',
             'dynamic': 'HL11/DynamicResults_HL11_run3-1.h5',
             'n_cond': 9},
    'HL13': {'n': 13, 'base_s': f'{_S}/Run4',  'base_d': f'{_D}/Run4',
             'static': 'HL13/StaticResults_HL13_run4-1.h5',
             'dynamic': 'HL13/DynamicResults_HL13_run4-1.h5',
             'n_cond': 9},
    'HL15': {'n': 15, 'base_s': f'{_S}/Run5',  'base_d': f'{_D}/Run5',
             'static': 'HL15/StaticResults_HL15_run5-1.h5',
             'dynamic': 'HL15/DynamicResults_HL15_run5-1.h5',
             'n_cond': 9},
    'HL17': {'n': 17, 'base_s': f'{_S}/Run6',  'base_d': f'{_D}/Run6',
             'static': 'HL17/StaticResults_HL17_run6-1.h5',
             'dynamic': 'HL17/DynamicResults_HL17_run6-1.h5',
             'n_cond': 9},
    'HL19': {'n': 19, 'base_s': f'{_S}/Run7',  'base_d': f'{_D}/Run7',
             'static': 'HL19/StaticResults_HL19_run7-1.h5',
             'dynamic': 'HL19/DynamicResults_HL19_run7-1.h5',
             'n_cond': 9},
    'HL21': {'n': 21, 'base_s': f'{_S}/Run8',  'base_d': f'{_D}/Run8',
             'static': 'HL21/StaticResults_HL21_run8-1.h5',
             'dynamic': 'HL21/DynamicResults_HL21_run8-1.h5',
             'n_cond': 9},
    'HL23': {'n': 23, 'base_s': f'{_S}/Run9',  'base_d': f'{_D}/Run9',
             'static': 'HL23/StaticResults_HL23_run9-1.h5',
             'dynamic': 'HL23/DynamicResults_HL23_run9-1.h5',
             'n_cond': 9},
    'HL25': {'n': 25, 'base_s': f'{_S}/Run10', 'base_d': f'{_D}/Run10',
             'static': 'HL25/StaticResults_HL25_run10-1.h5',
             'dynamic': 'HL25/DynamicResults_HL25_run10-1.h5',
             'n_cond': 8},
    'HL27': {'n': 27, 'base_s': f'{_S}/Run11', 'base_d': f'{_D}/Run11',
             'static': 'HL27/StaticResults_HL27_run11-1.h5',
             'dynamic': 'HL27/DynamicResults_HL27_run11-1.h5',
             'n_cond': 7},
    'HL29': {'n': 29, 'base_s': f'{_S}/Run12', 'base_d': f'{_D}/Run12',
             'static': 'HL29/StaticResults_HL29_run12-1.h5',
             'dynamic': 'HL29/DynamicResults_HL29_run12-1.h5',
             'n_cond': 7},
}

MOORINGS = [f'Mooring{i}' for i in range(1, 9)]
FENDERS  = [f'Fender{i}'  for i in range(1, 7)]


for cfg in LINES.values():
    nc      = cfg.get('n_cond', N_COND)
    Hs_vals = HS_VALUES[:nc]
    Tp_vals = np.sqrt(Hs_vals * 2 * np.pi * cfg['n'] / g)
    cfg['cond_params'] = {
        i + 1: (round(float(hs), 2), round(float(tp), 2))
        for i, (hs, tp) in enumerate(zip(Hs_vals, Tp_vals))
    }

def _resolve_paths(heading, cfg):
    s_nested = f'{heading}/{cfg["static"]}'
    if os.path.exists(s_nested):
        return s_nested, f'{heading}/{cfg["dynamic"]}'.rsplit('-', 1)[0]
    fname_s = cfg['static'].split('/', 1)[1]
    fname_d = cfg['dynamic'].split('/', 1)[1].rsplit('-', 1)[0]
    return f'{heading}/{fname_s}', f'{heading}/{fname_d}'

In [3]:
rows = []

for heading in HEADINGS:
    print(f'\n=== {heading} ===')
    for line_name, cfg in LINES.items():
        static_path, dyn_base = _resolve_paths(heading, cfg)
        dyn_files   = sorted(
            glob.glob(f'{dyn_base}-*.h5'),
            key=lambda p: int(p.rsplit('-', 1)[-1].replace('.h5', ''))
        )
        if not dyn_files:
            print(f'  {line_name}: no dynamic files found — skipping')
            continue
        print(f'  {line_name}: {len(dyn_files)} seed file(s)')

        run_s = cfg['base_s'].split('/')[-1]
        run_d = cfg['base_d'].split('/')[-1]

        static_by_cond = {}
        try:
            with h5py.File(static_path, 'r') as fs:
                base_s = f'{list(fs.keys())[0]}/{run_s}'
                for cond in fs[base_s].keys():
                    n     = int(cond.split('_')[-1])
                    grp_s = fs[f"{base_s}/{cond}/Static/Tanker"]
                    static_by_cond[n] = sum(
                        float(grp_s[f'{e}/FTotal'][()]) for e in MOORINGS
                    ) / 1e3
        except FileNotFoundError:
            print(f'    WARNING: static file not found — {static_path}')
            continue

        seed_stats = {n: {'means': [], 'moor_maxes': []} for n in static_by_cond}
        for dyn_file in dyn_files:
            try:
                with h5py.File(dyn_file, 'r') as fd:
                    base_d = f'{list(fd.keys())[0]}/{run_d}'
                    for cond in fd[base_d].keys():
                        n = int(cond.split('_')[-1])
                        if n not in seed_stats:
                            continue
                        grp_d   = fd[f"{base_d}/{cond}/Dynamic/Tanker/Positioning system force"]
                        moor_ts = sum(grp_d[f'TotalForce_{e}'][:] for e in MOORINGS) / 1e3
                        seed_stats[n]['means'].append(float(np.mean(moor_ts)))
                        seed_stats[n]['moor_maxes'].append(float(np.max(moor_ts)))
            except FileNotFoundError:
                print(f'    WARNING: could not open {dyn_file}')

        for n, stats in sorted(seed_stats.items()):
            means      = stats['means']
            moor_maxes = stats['moor_maxes']
            if not means:
                continue
            hs, tp = cfg['cond_params'].get(n, (np.nan, np.nan))
            rows.append({
                'heading':     heading,
                'line':        line_name,
                'cond_num':    n,
                'Hs':          hs,
                'Tp':          tp,
                'n_runs':      len(means),
                'static_kN':   static_by_cond[n],
                'dyn_mean_kN': float(np.mean(means)),
                'dyn_max_kN':  float(np.max(moor_maxes)),
                'dyn_std_kN':  float(np.std(moor_maxes, ddof=1)) if len(moor_maxes) > 1 else np.nan,
            })

df = pd.DataFrame(rows)
print(f'\nTotal rows: {len(df)}')
print(df.groupby(['heading', 'line'])[['n_runs']].first())


=== 0deg ===
  HL7: 5 seed file(s)


  HL9: 5 seed file(s)


  HL11: 5 seed file(s)


  HL13: 5 seed file(s)


  HL15: 5 seed file(s)


...

  HL23: 5 seed file(s)


  HL25: 5 seed file(s)


  HL27: 5 seed file(s)


  HL29: 5 seed file(s)



Total rows: 515
              n_runs
heading line        
0deg    HL11       5
        HL13       5
        HL15       5
        HL17       5
        HL19       5
        HL21       5
        HL23       5
        HL25       5
        HL27       5
        HL29       5
        HL7        5
        HL9        5
135deg  HL11       5
        HL13       5
        HL15       5
        HL17       5
        HL19       5
        HL21       5
        HL23       5
        HL25       5
        HL27       5
        HL29       5
        HL7        5
        HL9        5
180deg  HL11       5
        HL13       5
        HL15       5
        HL17       5
        HL19       5
        HL21       5
        HL23       5
        HL25       5
        HL27       5
        HL29       5
        HL7        5
        HL9        5
45deg   HL11       5
        HL13       5
        HL15       5
        HL17       5
        HL19       5
        HL21       5
        HL23       5
        HL25       5
        HL27     

In [4]:
out_path = 'statistics.csv'
df.to_csv(out_path, index=False, float_format='%.4f')
print(f'Saved → {out_path}')
display(df.round(2))

Saved → statistics.csv


,heading,line,cond_num,Hs,Tp,n_runs,static_kN,dyn_mean_kN,dyn_max_kN,dyn_std_kN
0,0deg,HL7,1,0.50,1.50,5,1216.65,1216.92,1217.95,0.05
1,0deg,HL7,2,0.83,1.93,5,1215.82,1216.11,1218.81,0.17
2,0deg,HL7,3,1.17,2.29,5,1214.62,1208.49,1219.14,0.82
3,0deg,HL7,4,1.50,2.59,5,1213.08,1180.97,1217.43,0.33
4,0deg,HL7,5,1.83,2.87,5,1211.54,1161.16,1217.11,1.38
...,...,...,...,...,...,...,...,...,...,...
510,180deg,HL29,3,1.17,4.66,5,1218.93,1218.83,1224.09,0.40
511,180deg,HL29,4,1.50,5.28,5,1219.19,1208.80,1226.57,0.72
512,180deg,HL29,5,1.83,5.84,5,1219.39,1211.80,1237.23,1.49
513,180deg,HL29,6,2.17,6.34,5,1219.46,1217.64,1256.63,2.56


## Per-element and vessel extraction

Produces three additional CSVs: `mooring_per_line.csv` is consumed by `08_DAF_Swells_vs_Steepness.ipynb`, `fender_statistics.csv` by `13_Fender_Capacity_Utilisation.ipynb`, and `vessel_statistics.csv` is kept for reference:

| File | Rows | Key columns |
|---|---|---|
| `mooring_per_line.csv` | line × cond × 8 moorings | `elem`, `static_kN`, `dyn_mean_kN`, `dyn_max_kN` |
| `fender_statistics.csv` | line × cond × 6 fenders | `elem`, `static_kN`, `dyn_mean_kN`, `dyn_max_kN` |
| `vessel_statistics.csv` | line × cond × 6 DOFs | `dof`, `static_val`, `dyn_mean`, `dyn_max` |

DOFs: `surge`/`sway`/`heave` in m, `roll`/`pitch`/`yaw` in deg.

In [5]:
FENDERS = [f'Fender{i}' for i in range(1, 7)]

DOF_MAP = {
    'surge': ('x',  'XGtranslationTotalmotion'),
    'sway':  ('y',  'YGtranslationTotalmotion'),
    'heave': ('z',  'ZGtranslationTotalmotion'),
    'roll':  ('rx', 'XLrotationTotalmotion'),
    'pitch': ('ry', 'YLrotationTotalmotion'),
    'yaw':   ('rz', 'ZGrotationTotalmotion'),
}

moor_rows   = []
fend_rows   = []
vessel_rows = []

for heading in HEADINGS:
    for line_name, cfg in LINES.items():
        static_path, dyn_base = _resolve_paths(heading, cfg)
        dyn_files   = sorted(
            glob.glob(f'{dyn_base}-*.h5'),
            key=lambda p: int(p.rsplit('-', 1)[-1].replace('.h5', ''))
        )
        if not dyn_files:
            continue

        run_s = cfg['base_s'].split('/')[-1]
        run_d = cfg['base_d'].split('/')[-1]

        moor_static   = {}
        fend_static   = {}
        vessel_static = {}

        try:
            with h5py.File(static_path, 'r') as fs:
                base_s = f'{list(fs.keys())[0]}/{run_s}'
                for cond in fs[base_s].keys():
                    n      = int(cond.split('_')[-1])
                    tanker = fs[f"{base_s}/{cond}/Static/Tanker"]
                    moor_static[n]   = {e: float(tanker[f'{e}/FTotal'][()]) / 1e3 for e in MOORINGS}
                    fend_static[n]   = {e: float(tanker[f'{e}/FTotal'][()]) / 1e3 for e in FENDERS}
                    sp = tanker['Static position']
                    vessel_static[n] = {dof: float(sp[sk][()]) for dof, (sk, _) in DOF_MAP.items()}
        except FileNotFoundError:
            print(f'WARNING: {static_path} not found')
            continue

        seed_data = {
            n: {'moor':   {e:   {'means': [], 'maxes': []} for e in MOORINGS},
                'fend':   {e:   {'means': [], 'maxes': []} for e in FENDERS},
                'vessel': {dof: {'means': [], 'maxes': []} for dof in DOF_MAP}}
            for n in moor_static
        }

        for dyn_file in dyn_files:
            try:
                with h5py.File(dyn_file, 'r') as fd:
                    base_d = f'{list(fd.keys())[0]}/{run_d}'
                    for cond in fd[base_d].keys():
                        n = int(cond.split('_')[-1])
                        if n not in seed_data:
                            continue
                        psf = fd[f"{base_d}/{cond}/Dynamic/Tanker/Positioning system force"]
                        gtp = fd[f"{base_d}/{cond}/Dynamic/Tanker/Global total position"]

                        for e in MOORINGS:
                            ts = psf[f'TotalForce_{e}'][:] / 1e3
                            seed_data[n]['moor'][e]['means'].append(float(np.mean(ts)))
                            seed_data[n]['moor'][e]['maxes'].append(float(np.max(ts)))
                        for e in FENDERS:
                            ts = psf[f'TotalForce_{e}'][:] / 1e3
                            seed_data[n]['fend'][e]['means'].append(float(np.mean(ts)))
                            seed_data[n]['fend'][e]['maxes'].append(float(np.max(ts)))
                        for dof, (_, hkey) in DOF_MAP.items():
                            ts = gtp[hkey][:]
                            seed_data[n]['vessel'][dof]['means'].append(float(np.mean(ts)))
                            seed_data[n]['vessel'][dof]['maxes'].append(float(np.max(ts)))
            except FileNotFoundError:
                print(f'WARNING: could not open {dyn_file}')

        for n in sorted(moor_static):
            sd     = seed_data[n]
            hs, tp = cfg['cond_params'].get(n, (np.nan, np.nan))
            base   = dict(heading=heading, line=line_name, cond_num=n, Hs=hs, Tp=tp)

            for e in MOORINGS:
                d = sd['moor'][e]
                moor_rows.append({**base, 'elem': e,
                    'static_kN':   moor_static[n][e],
                    'dyn_mean_kN': float(np.mean(d['means'])) if d['means'] else np.nan,
                    'dyn_max_kN':  float(np.max(d['maxes']))  if d['maxes'] else np.nan,
                    'dyn_std_kN':  float(np.std(d['maxes'], ddof=1)) if len(d['maxes']) > 1 else np.nan,
                })
            for e in FENDERS:
                d = sd['fend'][e]
                fend_rows.append({**base, 'elem': e,
                    'static_kN':   fend_static[n][e],
                    'dyn_mean_kN': float(np.mean(d['means'])) if d['means'] else np.nan,
                    'dyn_max_kN':  float(np.max(d['maxes']))  if d['maxes'] else np.nan,
                    'dyn_std_kN':  float(np.std(d['maxes'], ddof=1)) if len(d['maxes']) > 1 else np.nan,
                })
            for dof in DOF_MAP:
                d = sd['vessel'][dof]
                vessel_rows.append({**base, 'dof': dof,
                    'static_val': vessel_static[n][dof],
                    'dyn_mean':   float(np.mean(d['means'])) if d['means'] else np.nan,
                    'dyn_max':    float(np.max(d['maxes']))  if d['maxes'] else np.nan,
                    'dyn_std':    float(np.std(d['maxes'], ddof=1)) if len(d['maxes']) > 1 else np.nan,
                })

df_moor   = pd.DataFrame(moor_rows)
df_fend   = pd.DataFrame(fend_rows)
df_vessel = pd.DataFrame(vessel_rows)

for fname, df_out in [('mooring_per_line.csv', df_moor),
                      ('fender_statistics.csv', df_fend),
                      ('vessel_statistics.csv',  df_vessel)]:
    df_out.to_csv(fname, index=False, float_format='%.4f')
    print(f'Saved {fname}  ({len(df_out)} rows)')

Saved mooring_per_line.csv  (4120 rows)
Saved fender_statistics.csv  (3090 rows)
Saved vessel_statistics.csv  (3090 rows)
